In [ ]:
"""
Code to downsample uce.h5 and metadatafile to 4M (for UCE on 24GB GPU mem)
Adjust memory size based on input uce.h5 file size
"""

import numpy as np
import pandas as pd
import h5py
import argparse
import os, sys

emdedding_h5_file = "uce.h5"
emdedding_h5_file_key = "data"
metadata_file = "obs_anno.tsv"
max_cell_train = 4_000_000 # 4M for UCE on 24GB GPU mem

def density_weighted_sample(n_samples, knn_dists, alpha=1.0, random_state=42):
    """
    Sample n_samples indices, favoring sparse points.
    - knn_dists is given, it should be shape (N, k) of distances to k neighbors.
    - alpha controls strength: prob ∝ (mean_dist ** alpha). alpha>1 stronger bias to sparse.
    """
    rng = np.random.default_rng(random_state)
    if knn_dists is None:
        if emb is None:
            raise ValueError("Provide knn_dists or emb")
        from sklearn.neighbors import NearestNeighbors
        nbrs = NearestNeighbors(n_neighbors=16, n_jobs=-1).fit(emb)
        knn_dists = nbrs.kneighbors(emb, return_distance=True)[0]  # (N, k)
    # use mean distance to neighbors: larger => sparser
    mean_dist = knn_dists.mean(axis=1)
    # avoid zeros
    mean_dist = mean_dist + 1e-12
    probs = mean_dist ** float(alpha)
    probs = probs / probs.sum()
    sampled = rng.choice(len(mean_dist), size=n_samples, replace=False, p=probs)
    return sampled
    
p = argparse.ArgumentParser()
p.add_argument("--data_dir", 
                required=True,
                help="Specify data directory")
p.add_argument("--output_dir", 
                required=True,
                help="Specify output data directory")

args = p.parse_args()
results_dir = args.data_dir
output_dir = args.output_dir
network = args.network

if (network):
    # cp uce.h5 to local to increase speed
    remote_path = os.path.join(results_dir, "uce.h5")
    local_path = "/tmp/uce.h5"

    if not os.path.exists(local_path):
        print("Copying to local disk...")
        shutil.copy(remote_path, local_path)
    uce_path = local_path
else:
    uce_path = os.path.join(results_dir, "uce.h5")

# Load the embedding data
with h5py.File(uce_path, "r") as f:
    X_data = f[emdedding_h5_file_key]
    
n_rows = X_data.shape[0]    
if (n_rows <= max_cell_train):
    print ("cell number less than ",max_cell_train, "no need to downsampling")
    sys.exit(0)

with h5py.File(uce_path, "r") as f:
    X_data = f[emdedding_h5_file_key][:]
sampled_idx = density_weighted_sample(max_cell_train, knn_distances, alpha=1.0)
X_sampled = X_data[sampled_idx,:]

if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    
# export downsampled_data to disk
with h5py.File(os.path.join(output_dir, emdedding_h5_file), "w") as f:
    f.create_dataset(
        emdedding_h5_file_key,                # dataset name
        data=X_sampled,        # actual array
        compression="gzip",    # optional (smaller file)
        compression_opts=4,    # 1–9: higher = smaller + slower
        chunks=True            # enable chunking for flexibility
    )
    
# export metadata to disk
meta = pd.read_csv(metadata_file, sep="\t")
meta_sampled = meta.iloc[sampled_idx, :]
meta_sampled.to_csv(os.path.join(output_dir, metadata_file), sep="\t", index=False)